In [17]:
from pathlib import Path
from dotenv import load_dotenv
import os

env_path = Path.cwd().parent / ".env"
print(env_path.exists())

load_dotenv(env_path, override=True)
api_key = os.getenv("CFBD_API_KEY")
print(api_key is not None)

True
True


In [18]:
import requests
import pandas as pd

url = "https://api.collegefootballdata.com/plays"

headers = {
    "Authorization": f"Bearer {api_key}"
}

params = {
    "year": 2025,
    "week": 1,
    "team": "Indiana",
    "seasonType": "regular",
    "classification": "fbs"
}

In [19]:
response = requests.get(url, headers=headers, params=params)

print("Status code: ", response.status_code)

Status code:  200


In [20]:
plays = pd.DataFrame(response.json())

print("Shape: ", plays.shape)
print("Columns: ", plays.columns.to_list())

plays.head()

Shape:  (183, 27)
Columns:  ['gameId', 'driveId', 'id', 'driveNumber', 'playNumber', 'offense', 'offenseConference', 'offenseScore', 'defense', 'defenseConference', 'defenseScore', 'home', 'away', 'period', 'clock', 'offenseTimeouts', 'defenseTimeouts', 'yardline', 'yardsToGoal', 'down', 'distance', 'yardsGained', 'scoring', 'playType', 'playText', 'ppa', 'wallclock']


,gameId,driveId,id,driveNumber,playNumber,offense,offenseConference,offenseScore,defense,defenseConference,...,yardline,yardsToGoal,down,distance,yardsGained,scoring,playType,playText,ppa,wallclock
0,401752806,4017528065,401752806101979003,5,1,Old Dominion,Sun Belt,7,Indiana,Big Ten,...,66,66,1,10,0,False,Timeout,"Timeout Indiana, clock 02:09",NaN,2025-08-30T19:07:02.000Z
1,401752806,4017528065,401752806101995801,5,5,Old Dominion,Sun Belt,7,Indiana,Big Ten,...,60,60,3,4,-5,False,Penalty,"(00:41) PENALTY ODU False Start (Thomas,Ja'Cor...",NaN,2025-08-30T19:11:58.000Z
2,401752806,4017528065,401752806101999001,5,7,Old Dominion,Sun Belt,7,Indiana,Big Ten,...,65,65,4,9,91,True,Punt,"Ian Brandt punt for 56 yds , Jonathan Brady re...",NaN,2025-08-30T19:14:31.000Z
3,401752806,4017528065,401752806101999003,5,8,Old Dominion,Sun Belt,7,Indiana,Big Ten,...,75,75,1,10,0,False,Timeout,"Timeout Indiana, clock 00:09",NaN,2025-08-30T19:14:31.000Z
4,401752806,4017528065,401752806101984101,5,2,Old Dominion,Sun Belt,7,Indiana,Big Ten,...,80,80,1,10,14,False,Pass Reception,Colton Joseph pass complete to Tre' Brown III ...,1.049288,2025-08-30T19:19:12.000Z


In [21]:
plays["playType"].value_counts()

playType
Rush                          73
Pass Reception                29
Pass Incompletion             22
Timeout                       17
Penalty                        9
Punt                           9
Kickoff                        9
Rushing Touchdown              4
End Period                     2
Field Goal Good                2
Sack                           2
Field Goal Missed              1
End of Half                    1
Interception                   1
End of Game                    1
Fumble Recovery (Opponent)     1
Name: count, dtype: int64

In [22]:
plays.isnull().sum()

gameId                0
driveId               0
id                    0
driveNumber           0
playNumber            0
offense               0
offenseConference     0
offenseScore          0
defense               0
defenseConference     0
defenseScore          0
home                  0
away                  0
period                0
clock                 0
offenseTimeouts       0
defenseTimeouts       0
yardline              0
yardsToGoal           0
down                  0
distance              0
yardsGained           0
scoring               0
playType              0
playText              0
ppa                  51
wallclock             0
dtype: int64

In [23]:
plays[["down", "distance", "yardsToGoal", "offenseScore", "defenseScore", "playType", "yardsGained"]].head(10)

,down,distance,yardsToGoal,offenseScore,defenseScore,playType,yardsGained
0,1,10,66,7,0,Timeout,0
1,3,4,60,7,0,Penalty,-5
2,4,9,65,7,7,Punt,91
3,1,10,75,7,7,Timeout,0
4,1,10,80,7,0,Pass Reception,14
5,1,10,66,7,0,Pass Reception,3
6,2,7,63,7,0,Rush,3
7,3,9,65,7,0,Pass Incompletion,0
8,1,10,65,7,7,Kickoff,0
9,0,10,75,7,7,End Period,9


In [24]:
plays[["playType","playText"]].head(25)

,playType,playText
0,Timeout,"Timeout Indiana, clock 02:09"
1,Penalty,"(00:41) PENALTY ODU False Start (Thomas,Ja'Cor..."
2,Punt,"Ian Brandt punt for 56 yds , Jonathan Brady re..."
3,Timeout,"Timeout Indiana, clock 00:09"
4,Pass Reception,Colton Joseph pass complete to Tre' Brown III ...
5,Pass Reception,Colton Joseph pass complete to Tre' Brown III ...
6,Rush,Colton Joseph run for 3 yds to the ODU 40
7,Pass Incompletion,Colton Joseph pass incomplete to Trequan Jones
8,Kickoff,Brendan Franke kickoff for 65 yds for a touchback
9,End Period,End of 1st Quarter


In [26]:
pass_run_types = [
    "Rush",
    "Rushing Touchdown",
    "Pass Reception",
    "Pass Incompletion",
    "Sack"
]

plays.loc[
    plays["playType"].isin(pass_run_types),
    ["down", "distance", "yardsToGoal", "playType", "playText"]
].head(20)

,down,distance,yardsToGoal,playType,playText
4,1,10,80,Pass Reception,Colton Joseph pass complete to Tre' Brown III ...
5,1,10,66,Pass Reception,Colton Joseph pass complete to Tre' Brown III ...
6,2,7,63,Rush,Colton Joseph run for 3 yds to the ODU 40
7,3,9,65,Pass Incompletion,Colton Joseph pass incomplete to Trequan Jones
11,1,10,75,Rush,Colton Joseph run for 9 yds to the ODU 34
12,2,1,66,Pass Incompletion,Colton Joseph pass incomplete to Tre' Brown III
13,3,1,66,Rush,Colton Joseph run for 3 yds to the ODU 37 for ...
14,1,10,63,Rush,Maurki James run for no gain to the ODU 37
17,1,10,55,Rush,Roman Hemby run for 14 yds to the ODU 41 for a...
18,1,10,41,Pass Reception,Fernando Mendoza pass complete to E.J. William...


In [27]:
plays['play_call'] =  plays['playType'].map({
    'Rush': 'Run',
    'Rushing Touchdown': 'Run',
    'Pass Reception': 'Pass',
    'Pass Incompletion': 'Pass',
    'Sack': 'Pass'
})

plays[["playType", "play_call"]].head(20)

,playType,play_call
0,Timeout,NaN
1,Penalty,NaN
2,Punt,NaN
3,Timeout,NaN
4,Pass Reception,Pass
5,Pass Reception,Pass
6,Rush,Run
7,Pass Incompletion,Pass
8,Kickoff,NaN
9,End Period,NaN


In [28]:
plays["play_call"].value_counts(dropna=False)

play_call
Run     77
NaN     53
Pass    53
Name: count, dtype: int64

In [25]:
params["team"] = "Georgia"

response = requests.get(url, headers=headers, params=params)
print(response.status_code)

georgia_plays = pd.DataFrame(response.json())
print(georgia_plays.shape)

200
(173, 27)


In [29]:
teams = ["Indiana", "Georgia"]
weeks = range(1, 17)

team_data = {}

for team in teams:
    weekly_data = []
    for week in weeks:
        params = {
            "year": 2025,
            "week": week,
            "team": team,
            "seasonType": "regular",
            "classification": "fbs"
        }

        response = requests.get(url, headers=headers, params=params)
        response.raise_for_status()

        week_plays = pd.DataFrame(response.json())

        if not week_plays.empty:
            week_plays['requested_team'] = team
            weekly_data.append(week_plays)
    if weekly_data:
        team_data[team] = pd.concat(weekly_data, ignore_index=True)

    print(team, team_data[team].shape)

Indiana (2188, 28)
Georgia (2228, 28)


In [30]:
indiana_plays = team_data["Indiana"]
georgia_plays = team_data["Georgia"]

print("Indiana plays shape: ", indiana_plays.shape)
print("Georgia plays shape: ", georgia_plays.shape)

print(indiana_plays["gameId"].nunique(), "Indiana-related games")
print(georgia_plays["gameId"].nunique(), "Georgia-related games")

Indiana plays shape:  (2188, 28)
Georgia plays shape:  (2228, 28)
13 Indiana-related games
13 Georgia-related games


In [31]:
print(indiana_plays["offense"].value_counts().head())
print(indiana_plays["defense"].value_counts().head())

print(georgia_plays["offense"].value_counts().head())
print(georgia_plays["defense"].value_counts().head())

offense
Indiana           1194
Penn State          96
Purdue              95
Michigan State      86
Iowa                85
Name: count, dtype: int64
defense
Indiana          994
Old Dominion     115
Oregon           105
UCLA             105
Indiana State     96
Name: count, dtype: int64
offense
Georgia      1191
Alabama       175
Tennessee      93
Auburn         89
Kentucky       84
Name: count, dtype: int64
defense
Georgia      1037
Alabama       153
Tennessee     113
Ole Miss      104
Marshall       98
Name: count, dtype: int64


In [32]:
from pathlib import Path

project_dir = Path.cwd().parent

raw_dir = project_dir / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

indiana_plays.to_csv(raw_dir / "indiana_plays.csv", index=False)
georgia_plays.to_csv(raw_dir / "georgia_plays.csv", index=False)

In [34]:
all_plays = pd.concat([indiana_plays, georgia_plays], ignore_index=True)

print("All plays shape: ", all_plays.shape)
print("Unique plays: ", all_plays["id"].nunique())
print("Team requested: ", all_plays["requested_team"].value_counts())

all_plays.to_csv(raw_dir / "all_plays.csv", index=False)

All plays shape:  (4416, 28)
Unique plays:  4416
Team requested:  requested_team
Georgia    2228
Indiana    2188
Name: count, dtype: int64
